In [136]:
import pandas as pd
from pathlib import Path
import altair as alt

In [137]:
IN = 'listings.csv'
OUT = 'cleaned_listings.csv'

In [138]:
df = pd.read_csv(IN)

In [139]:
df = df[['price', 'number_of_reviews', 'latitude', 'longitude', 'room_type', 'minimum_nights', 'availability_365', 'review_scores_rating', 'license', 'neighbourhood_cleansed', 'calculated_host_listings_count', 'bedrooms', 'name']]
df = df.dropna(subset = [col for col in df.columns if col != 'license'])
df['price'] = df['price'].str.replace("[$,]", "", regex=True).astype(float)
df['review_scores_rating'] = df['review_scores_rating'].astype(float)
df['has_license'] = df['license'].notna().astype(int)
df['license'] = df['license'].fillna('none')

In [140]:
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print(f"Done! -> {OUT}  rows={len(df)}  cols={df.shape[1]}")

Done! -> cleaned_listings.csv  rows=2779  cols=14


# First Visualization
## Histogram of minimum nights and its relationship with availability


In [141]:
# filter the data to exclude outliers
df_filtered1 = df[df['minimum_nights'] <= 15].copy()
# Create bins for availability_365 to show the relationship
df_filtered1['availability_category'] = pd.cut(
    df_filtered1['availability_365'], 
    bins=[0, 90, 180, 270, 365],
    labels=['Low (0-90)', 'Medium (91-180)', 'High (181-270)', 'Very High (271-365)']
)

In [142]:
alt.Chart(df_filtered).mark_bar(opacity=0.7).encode(
    x=alt.X('minimum_nights:Q', 
            bin=alt.Bin(maxbins=20),
            title='Minimum Nights Required'),
    y=alt.Y('count()', title='Number of Listings'),
    color=alt.Color('availability_category:N',
                    title='Availability (365 days)',
                    scale=alt.Scale(scheme='viridis')),
    tooltip=[
        alt.Tooltip('minimum_nights:Q', bin=True, title='Minimum Nights'),
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('availability_category:N', title='Availability Level')
    ]
).properties(
    width=700,
    height=400,
    title='Distribution of Minimum Nights by Availability Level'
).interactive()


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.Chart(...)

# Second Visualization
##  Scatter plot - Price vs. Reviews with neighborhood coloring

In [143]:
df_filtered2 = df[
    (df['price'] > 0) & 
    (df['price'] < 1000) &
    (df['number_of_reviews'] < 300)
].copy()

In [145]:
alt.Chart(df_filtered).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('price:Q', 
            title='Price per Night ($)',
            scale=alt.Scale(domain=[0, 1000])),
    y=alt.Y('number_of_reviews:Q', 
            title='Number of Reviews'),
    color=alt.Color('neighbourhood_cleansed:N', 
                    title='Neighborhood',
                    scale=alt.Scale(scheme='tableau20')),
    tooltip=[
        alt.Tooltip('neighbourhood_cleansed:N', title='Neighborhood'),
        alt.Tooltip('price:Q', title='Price', format='$.2f'),
        alt.Tooltip('number_of_reviews:Q', title='Reviews'),
        alt.Tooltip('room_type:N', title='Room Type')
    ]
).properties(
    width=900,
    height=600,
    title='Price vs. Number of Reviews by Neighborhood'
).interactive()

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.Chart(...)